# CommuteCast — Stage 4: Polynomial Curve Fitting

## Unit-1 Conceptual Foundations:
- **Goal**: Model the non-linear relationship between $X = \text{Hour of Day}$ ($0$ to $23$) and $Y = \text{Mean Traffic Volume}$ (vehicles/hour).
- **Objective**: Estimate empirical expected traffic volume for any given hour using Ordinary Least Squares (OLS) polynomial curves.
- **Degrees Evaluated**: Degree 1 (Linear), Degree 2 (Quadratic), and Degree 3 (Cubic).
- **Model Selection Criteria**: Balancing goodness-of-fit ($R^2$, MAE, RMSE) with parsimony and avoidance of overfitting.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))
from src.polynomial import (
    prepare_hourly_data,
    fit_polynomial_models,
    evaluate_polynomial_models,
    predict_traffic_for_hour,
    get_default_predictor
)
from src.visualizations import plot_stage4_polynomial_fits, plot_stage4_residuals

# Load cleaned preprocessed dataset
df = pd.read_csv('../data/preprocessed_traffic_volume.csv')
print(f'Cleaned dataset loaded with {len(df):,} observations.')

## 1. Hourly Aggregation

We aggregate observations across the entire multi-year historical dataset to calculate the empirical sample mean for each hour $h \in \{0, 1, \dots, 23\}$:
$$\bar{y}_h = \frac{1}{N_h}\sum_{i=1}^{N_h} y_{h, i}$$

This yields exactly 24 observations.

In [ ]:
hourly_df = prepare_hourly_data(df)
display(hourly_df)

## 2. Polynomial Model Fitting (Degrees 1, 2, 3)

- **Degree 1 (Linear)**: $y = a_0 + a_1 x$
- **Degree 2 (Quadratic)**: $y = a_0 + a_1 x + a_2 x^2$
- **Degree 3 (Cubic)**: $y = a_0 + a_1 x + a_2 x^2 + a_3 x^3$

In [ ]:
X = hourly_df['hour'].values
y = hourly_df['mean_traffic_volume'].values

models = fit_polynomial_models(X, y, degrees=(1, 2, 3))
for deg in [1, 2, 3]:
    print(f'Degree {deg}: {models[deg]["equation_str"]}')

## 3. Model Evaluation: MAE, RMSE, and $R^2$

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|$$
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2}$$
$$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

In [ ]:
eval_df = evaluate_polynomial_models(models, X, y)
display(eval_df)

## 4. Model Selection & Tradeoff Analysis

- **Degree 1 ($R^2 = 16.06\%$, $\text{MAE} = 1,505.0$)**: Fails dramatically. A linear function cannot capture the rise and fall of daytime travel.
- **Degree 2 ($R^2 = 83.52\%$, $\text{MAE} = 554.2$)**: Captures the broad diurnal curve, but forces a symmetric parabolic peak centered at 1:00 PM, severely underestimating the evening commute.
- **Degree 3 ($R^2 = 85.75\%$, $\text{MAE} = 539.6$)**: Introduces an inflection point that fits the late afternoon surge much better, providing the lowest error without risk of overfitting on 24 points.

**Selected Degree**: **Degree 3**.

## 5. Visualizations

In [ ]:
# Curve fits comparison
fig1 = plot_stage4_polynomial_fits(hourly_df, models)
plt.show()

In [ ]:
# Residuals analysis
fig2 = plot_stage4_residuals(hourly_df, models)
plt.show()

## 6. Example Hourly Predictions

Evaluating predictions for the key commute hours:

In [ ]:
hours_to_test = [7, 8, 16, 17]
pred_table = []
for h in hours_to_test:
    actual = hourly_df.loc[hourly_df['hour'] == h, 'mean_traffic_volume'].values[0]
    p1 = predict_traffic_for_hour(h, degree=1)
    p2 = predict_traffic_for_hour(h, degree=2)
    p3 = predict_traffic_for_hour(h, degree=3)
    pred_table.append({
        'Hour': f'{h:02d}:00',
        'Actual Mean Volume': actual,
        'Degree 1 Prediction': p1,
        'Degree 2 Prediction': p2,
        'Degree 3 Prediction (Selected)': p3,
        'Degree 3 Error': actual - p3
    })
display(pd.DataFrame(pred_table))